# F1-M01: Reportes Sweetviz — Datos Crudos

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 ¿Qué hace este notebook?

Genera **reportes Sweetviz** de los datos **ANTES** de limpiar (datos crudos).
Sweetviz es una librería que crea automáticamente un informe HTML interactivo
con estadísticas de cada columna: distribución, nulos, tipos, correlaciones, etc.

Estos reportes sirven como **línea base**: permiten comparar cómo estaban
los datos antes y después de la limpieza (Fase 1, módulo M03).

## ⚠️ Requisitos

- Haber ejecutado `00_configuracion_proyecto.ipynb` (crea las carpetas)
- Los 2 archivos Excel originales en `data/00_raw/`
- La librería `sweetviz` instalada (`pip install sweetviz`)

## 📦 ¿Qué genera?

| Archivo | Ubicación | Descripción |
|---|---|---|
| 9 reportes Sweetviz | `docs/html/fase1/reportes_raw/` | Uno por cada tabla del Excel |
| m01_reportes_raw.html | `docs/html/fase1/` | Página índice con KPIs y enlaces a reportes |

## 📁 Flujo de datos

```
data/00_raw/datos_proyecto_sin_preinscrip.xlsx  →  8 reportes Sweetviz
data/00_raw/preinscripcion_si.xlsx              →  1 reporte Sweetviz
```

## 📌 Siguiente

- `f1_m02_limpieza.ipynb` (limpieza de datos)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Esta celda hace 3 cosas:
#   1. Importa las librerías necesarias (pandas, numpy, sweetviz)
#   2. Detecta el entorno (Colab o Local) y encuentra la raíz del proyecto
#   3. Importa las rutas y funciones de src.config (NUNCA hardcodeadas)
#
# El truco de numpy.VisibleDeprecationWarning es necesario porque Sweetviz
# usa una versión antigua de numpy internamente y sin esto da un warning.
# ============================================================================

import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

# Silenciar warnings molestos (Sweetviz genera muchos)
warnings.filterwarnings('ignore')

# --- Fix para numpy + Sweetviz ---
# Sweetviz internamente usa np.VisibleDeprecationWarning, pero en versiones
# recientes de numpy (>=2.0) lo movieron a np.exceptions.
# Este parche evita que Sweetviz falle al importarse.
if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning

# --- Importar Sweetviz ---
import sweetviz as sv
print(f'✅ Sweetviz {sv.__version__}')

# --- Detectar entorno ---
# En Colab: monta Google Drive y usa la ruta fija de AU_UJI
# En Local: sube niveles desde la carpeta actual hasta encontrar 'src/'
#           Esto funciona desde cualquier carpeta dentro del proyecto.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

# Verificar que encontramos la raíz del proyecto
if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
# Todas las rutas vienen de src.config (definidas en config_entorno.py).
# MAPEO_HOJAS es un diccionario que traduce nombres de hojas del Excel
# a nombres internos: {'Titulaciones': 'titulaciones', 'Recibos': 'recibos', ...}
from src.config import (
    RUTA_RAW, RUTA_HTML,
    EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION,
    MAPEO_HOJAS,
    info_entorno
)

# Funciones de utilidad del proyecto
from src.utils import crear_directorios, formato_numero_es

# Funciones para generar HTML (KPIs, tarjetas, tablas, navegación)
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    generar_tabla_con_tooltip,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Crear carpetas de salida ---
# RUTA_FASE1: donde van las páginas HTML de la Fase 1
# RUTA_REPORTES_RAW: subcarpeta para los reportes Sweetviz individuales
RUTA_FASE1 = RUTA_HTML / 'fase1'
RUTA_REPORTES_RAW = RUTA_FASE1 / 'reportes_raw'
crear_directorios([RUTA_FASE1, RUTA_REPORTES_RAW])

# Atajo para formatear números en español (1.234.567 en vez de 1,234,567)
fmt = formato_numero_es

# Mostrar información del entorno
info_entorno()

✅ Sweetviz 2.3.1
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS RAW (desde los Excel originales)
# ============================================================================
#
# Lee las 8 hojas del Excel principal + 1 hoja de preinscripción.
# Los datos NO se transforman aquí, se leen tal cual están en el Excel.
#
# MAPEO_HOJAS traduce nombres de hojas a nombres internos:
#   'Nac-Sexo_Nacionalidad' → 'demograficos'
#   'Circunstancias'        → 'becas'
#   etc.
#
# El resultado es un diccionario: dfs_raw = {'titulaciones': df, 'recibos': df, ...}
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS RAW')
print('=' * 60)

# Diccionario donde guardaremos cada tabla como un DataFrame
dfs_raw = {}

# --- Excel Principal (8 hojas) ---
# Leemos hoja por hoja usando el mapeo de nombres
print(f'\n📖 Leyendo {EXCEL_PRINCIPAL.name}...')
for hoja_excel, nombre_interno in MAPEO_HOJAS.items():
    try:
        df = pd.read_excel(EXCEL_PRINCIPAL, sheet_name=hoja_excel)
        dfs_raw[nombre_interno] = df
        print(f'   ✅ {hoja_excel:30s} → {nombre_interno:15s}: {len(df):>8,} filas × {len(df.columns)} cols')
    except Exception as e:
        print(f'   ❌ Error en {hoja_excel}: {e}')

# --- Excel Preinscripción (1 hoja) ---
# Este archivo tiene solo 1 hoja con los datos de preinscripción
print(f'\n📖 Leyendo {EXCEL_PREINSCRIPCION.name}...')
try:
    df = pd.read_excel(EXCEL_PREINSCRIPCION)
    dfs_raw['preinscripcion'] = df
    print(f'   ✅ {"preinscripcion":30s}   {"":15s}: {len(df):>8,} filas × {len(df.columns)} cols')
except Exception as e:
    print(f'   ❌ Error: {e}')

print(f'\n✅ Total: {len(dfs_raw)} tablas cargadas')

CARGANDO DATOS RAW

📖 Leyendo datos_proyecto_sin_preinscrip.xlsx...
   ✅ Titulaciones                   → titulaciones   :       45 filas × 6 cols
   ✅ Recibos                        → recibos        :  114,454 filas × 5 cols
   ✅ Domicilios                     → domicilios     :  210,911 filas × 6 cols
   ✅ Expedientes                    → expedientes    :  109,568 filas × 15 cols
   ✅ Nac-Sexo_Nacionalidad          → demograficos   :   30,873 filas × 5 cols
   ✅ Circunstancias                 → becas          :   70,524 filas × 4 cols
   ✅ Trabajo                        → trabajo        :  195,524 filas × 4 cols
   ✅ Notas                          → notas          :  107,908 filas × 5 cols

📖 Leyendo preinscripcion_si.xlsx...
   ✅ preinscripcion                                  :  210,986 filas × 24 cols

✅ Total: 9 tablas cargadas


In [3]:
# ============================================================================
# CELDA 3: GENERAR REPORTES SWEETVIZ
# ============================================================================
#
# Para cada tabla, genera un reporte Sweetviz (archivo HTML independiente).
# Sweetviz analiza automáticamente cada columna y muestra:
#   - Distribución de valores
#   - Porcentaje de nulos
#   - Estadísticas (media, mediana, etc.)
#   - Correlaciones entre variables
#
# También calculamos métricas básicas (filas, columnas, nulos, duplicados)
# que usaremos luego para los KPIs y la tabla resumen del HTML.
#
# NOTA: Sweetviz no acepta columnas con tipos mixtos (números y texto
# mezclados), así que convertimos las columnas 'object' a string.
#
# ⏱️ Esta celda tarda unos minutos (genera 9 reportes).
# ============================================================================

print('=' * 60)
print('GENERANDO REPORTES SWEETVIZ')
print('=' * 60)

# Lista donde guardamos las métricas de cada tabla
# (la usaremos después para generar la tabla HTML resumen)
metricas_tablas = []

for nombre, df in dfs_raw.items():
    print(f'\n📊 Generando reporte: {nombre}...')

    # --- Calcular métricas de calidad ---
    n_filas = len(df)
    n_cols = len(df.columns)
    n_nulos = df.isna().sum().sum()       # total de celdas vacías
    total_celdas = n_filas * n_cols        # total de celdas en la tabla
    pct_nulos = (n_nulos / total_celdas * 100) if total_celdas > 0 else 0
    n_duplicados = df.duplicated().sum()   # filas duplicadas exactas

    metricas_tablas.append({
        'nombre': nombre.capitalize(),
        'filas': n_filas,
        'columnas': n_cols,
        'nulos_pct': f'{pct_nulos:.2f}%' if pct_nulos > 0 else '0.0%',
        'duplicados': n_duplicados,
        'cols_lista': list(df.columns)  # para el tooltip de la tabla HTML
    })

    # --- Preparar datos para Sweetviz ---
    # Sweetviz falla con columnas de tipo 'object' que mezclan números y texto.
    # Solución: convertir todas las columnas 'object' a string puro.
    df_sv = df.copy()
    for col in df_sv.columns:
        if df_sv[col].dtype == 'object':
            df_sv[col] = df_sv[col].astype(str)

    # --- Generar y guardar el reporte Sweetviz ---
    # pairwise_analysis='off' desactiva las correlaciones entre pares
    # (mucho más rápido, y con datos crudos las correlaciones no son fiables)
    ruta_reporte = RUTA_REPORTES_RAW / f'reporte_{nombre}.html'

    try:
        report = sv.analyze(df_sv, pairwise_analysis='off')
        report.show_html(filepath=str(ruta_reporte), open_browser=False)
        print(f'   ✅ {ruta_reporte.name}')
    except Exception as e:
        print(f'   ❌ Error: {e}')

print(f'\n✅ {len(metricas_tablas)} reportes generados en {RUTA_REPORTES_RAW}')

GENERANDO REPORTES SWEETVIZ

📊 Generando reporte: titulaciones...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw\reporte_titulaciones.html was generated.
   ✅ reporte_titulaciones.html

📊 Generando reporte: recibos...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw\reporte_recibos.html was generated.
   ✅ reporte_recibos.html

📊 Generando reporte: domicilios...


...

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw\reporte_trabajo.html was generated.
   ✅ reporte_trabajo.html

📊 Generando reporte: notas...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw\reporte_notas.html was generated.
   ✅ reporte_notas.html

📊 Generando reporte: preinscripcion...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw\reporte_preinscripcion.html was generated.
   ✅ reporte_preinscripcion.html

✅ 9 reportes generados en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw


In [4]:
# ============================================================================
# CELDA 4: CALCULAR KPIs (indicadores clave)
# ============================================================================
#
# Los KPIs son los números grandes que aparecen arriba de la página HTML.
# Se calculan sumando las métricas de todas las tablas.
#
# Usamos formato_numero_es (importado como 'fmt') para que los números
# se muestren en formato español: 1.050.808 en vez de 1,050,808
# ============================================================================

# Sumar totales de todas las tablas
total_filas = sum(m['filas'] for m in metricas_tablas)
total_cols = sum(m['columnas'] for m in metricas_tablas)
total_nulos = sum(df.isna().sum().sum() for df in dfs_raw.values())

# Definir los KPIs que se mostrarán en la página HTML
# Cada KPI tiene: valor (el número), titulo (la etiqueta), color (para el HTML)
KPIS = [
    {'valor': str(len(dfs_raw)),  'titulo': 'Tablas',        'color': '#3182ce'},  # azul
    {'valor': fmt(total_filas),   'titulo': 'Registros',     'color': '#38a169'},  # verde
    {'valor': str(total_cols),    'titulo': 'Columnas',      'color': '#ed8936'},  # naranja
    {'valor': fmt(total_nulos),   'titulo': 'Valores nulos', 'color': '#e53e3e'},  # rojo
]

print('KPIs calculados:')
for kpi in KPIS:
    print(f"  {kpi['titulo']:15s}: {kpi['valor']}")

KPIs calculados:
  Tablas         : 9
  Registros      : 1.050.793
  Columnas       : 74
  Valores nulos  : 232.495


In [5]:
# ============================================================================
# CELDA 5: PREPARAR TARJETAS DE REPORTES
# ============================================================================
#
# Las "tarjetas" son los recuadros clicables que aparecen en la página HTML.
# Cada tarjeta enlaza a un reporte Sweetviz individual.
#
# Los colores se asignan rotativamente de una paleta predefinida.
# Ejemplo: si hay 9 tablas y 6 colores, se repiten desde el principio.
# ============================================================================

# Paleta de colores para las tarjetas (se repiten si hay más tablas que colores)
PALETA_TARJETAS = ['#3182ce', '#805ad5', '#e53e3e', '#38a169', '#ed8936', '#319795']

TARJETAS_REPORTES = []
for i, m in enumerate(metricas_tablas):
    color = PALETA_TARJETAS[i % len(PALETA_TARJETAS)]  # color rotativo
    TARJETAS_REPORTES.append({
        'titulo': m['nombre'],
        'descripcion': f"{fmt(m['filas'])} filas | {m['columnas']} cols | {m['nulos_pct']} nulos",
        'emoji': '📋',
        'link': f"reportes_raw/reporte_{m['nombre'].lower()}.html",
        'link_texto': 'Ver Sweetviz →',
        'color': color
    })

print(f'✅ {len(TARJETAS_REPORTES)} tarjetas preparadas')

✅ 9 tarjetas preparadas


In [6]:
# ============================================================================
# CELDA 6: GENERAR PÁGINA HTML
# ============================================================================
#
# Construye la página m01_reportes_raw.html con:
#   - Navegación por fases y módulos (generada dinámicamente)
#   - KPIs (los números grandes de arriba)
#   - Tarjetas clicables que enlazan a cada reporte Sweetviz
#   - Tabla resumen con métricas de calidad de cada tabla
#
# render_base_html() usa la plantilla base.html (Jinja2) que incluye:
#   - Cabecera con logos UOC/UJI
#   - Footer con datos de la autora (desde config_proyecto.py)
#   - Enlace al notebook en GitHub
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación dinámica ---
# Genera los menús de fases (arriba) y módulos (debajo)
# fase_activa='fase1' resalta la Fase 1 en el menú
# modulo_activo='m01' resalta el módulo M01
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase1',
    modulo_activo='m01'
)

# --- KPIs (números grandes de arriba) ---
kpis_html = generar_kpis_html(KPIS)

# --- Sección: Tarjetas de reportes ---
# Cada tarjeta es un enlace a un reporte Sweetviz individual
tarjetas_html = generar_tarjetas_html(TARJETAS_REPORTES)
seccion_reportes = generar_seccion_html(
    titulo='Reportes Sweetviz — Datos ANTES de limpiar',
    contenido=tarjetas_html,
    icono='📊'
)

# --- Sección: Tabla resumen de calidad ---
# Muestra filas, columnas, % nulos y duplicados de cada tabla
# Los tooltips muestran los nombres de las columnas al pasar el ratón
tabla_html = generar_tabla_con_tooltip(metricas_tablas)
seccion_tabla = generar_seccion_html(
    titulo='Resumen de Calidad',
    contenido=tabla_html,
    icono='📋'
)

# --- Juntar todo el contenido ---
contenido_html = kpis_html + seccion_reportes + seccion_tabla

# --- Generar página completa con plantilla base ---
html_completo = render_base_html(
    titulo='M01: Reportes Raw',
    icono='📋',
    subtitulo='Fase 1: Transformación | TFM Abandono Universitario',
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f1_m01_reportes_raw.ipynb',
    notebook_carpeta='fase1_transformacion'
)

# --- Guardar en disco ---
ruta_html = RUTA_FASE1 / 'm01_reportes_raw.html'
guardar_html(html_completo, ruta_html)

print(f'\n✅ HTML generado: {ruta_html}')

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m01_reportes_raw.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m01_reportes_raw.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================
#
# Muestra un resumen de todo lo generado en este notebook.
# Útil para verificar que todo se ejecutó correctamente.
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M01 COMPLETADO')
print('=' * 60)
print(f'\n📁 Archivos generados:')
print(f'   📄 {ruta_html}')
print(f'   📊 {len(metricas_tablas)} reportes Sweetviz en {RUTA_REPORTES_RAW}')
print(f'\n📊 KPIs: {len(dfs_raw)} tablas, {fmt(total_filas)} registros, {total_cols} columnas, {fmt(total_nulos)} nulos')
print(f'\n📌 Siguiente: f1_m02_limpieza.ipynb')
print('=' * 60)


✅ F1-M01 COMPLETADO

📁 Archivos generados:
   📄 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m01_reportes_raw.html
   📊 9 reportes Sweetviz en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_raw

📊 KPIs: 9 tablas, 1.050.793 registros, 74 columnas, 232.495 nulos

📌 Siguiente: f1_m02_limpieza.ipynb
